Import dataset and turn to a dataframe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gensim
import pandas as pd
import numpy as np
import glob
import json
import gc

SAVE_DIR = '/content/drive/MyDrive/dblp-ref'

from google.colab import drive
drive.mount('/content/drive')

compression_step = True
if not compression_step:
    # Extract the dataset and put folder into Google Drive.
    # MyDrive/dblp-ref/dflp-ref-*.json
    print("Files not compressed. Loading uncompressed files from Drive.")
    json_files = glob.glob(SAVE_DIR + '/dblp-ref-*.json')
    print(f"Found JSON files: {json_files}")
    df_list = []

    for file in json_files:
        print(f"Processing file: {file}")
        data = []
        with open(file, 'r') as f:
            for line_num, line in enumerate(f):
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Skipping malformed JSON line {line_num+1} in {file}: {e}")
                    continue

        if data:
            df = pd.DataFrame(data)
            df_list.append(df)
        else:
            print(f"No valid JSON data found in {file}")
            exit(1)

    # Concatenate all dataframes into a single dataframe
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        display(combined_df.head())
        combined_df.to_parquet(SAVE_DIR + "/papers.parquet", index=False)
        print("Compressed file loaded to drive.")
    else:
        print("No dataframes were created due to errors or empty files.")
        exit(1)
else:
    print("Compressed flag set true. Fetching compressed file.")
    combined_df = pd.read_parquet(SAVE_DIR + "/papers.parquet")
    print("Compressed parquet file loaded")
    display(combined_df.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.9 MB/s eta 0:00:00
Mounted at /content/drive
Compressed flag set true. Fetching compressed file.


## Find Missing Values

In [ ]:
combined_df.isnull().sum()

NameError: name 'combined_df' is not defined

In [ ]:
def missing_values_table(df):
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table_ren_columns = mis_val_table.rename(
    columns = {0: "Missing Values", 1: "% of Total Values"})

    return mis_val_table_ren_columns

table = missing_values_table(combined_df)

table.round(2)


---
# Data Preprocessing
The following steps clean and prepare the dataset for EDA, Clustering, and Classification.

**Summary of steps:**
1. Fill missing values with safe defaults
2. Normalize and clean text (title + abstract)
3. Filter noisy venues with too few papers
4. Engineer new numerical features from existing columns
5. Delete unneccesary columns
6. Impute numeric missing values with medians and apply Z-score normalization
7. TF-IDF vectorization of text
8. Dimensionality reduction with TruncatedSVD (for EDA scatter plots)

### Preprocessing Step 1: Fill Missing Values
- `abstract`: fill with empty string `""` so TF-IDF still works on these rows
- `references` and `authors`: fill with empty list `[]` since they are list columns
- We also add a flag column `has_abstract` so we can choose to exclude these rows in NLP tasks if needed

In [ ]:
# Fill missing abstracts with empty string
combined_df['abstract'] = combined_df['abstract'].fillna('')
combined_df['references'] = combined_df['references'].apply(
    lambda x: list() if x is None or isinstance(x, float) else list(x)
)
combined_df['authors'] = combined_df['authors'].apply(
    lambda x: list() if x is None or isinstance(x, float) else list(x)
)

print(f"Total papers: {len(combined_df):,}")
print("\nRemaining missing values:")
print(combined_df.isnull().sum())

### Preprocessing Step 2: Normalize and Clean Text
We clean the `title` and `abstract` columns by:
- Converting to lowercase
- Removing special characters (keeping only letters, numbers, spaces)
- Normalizing extra whitespace

We then combine `clean_title` + `clean_abstract` into a single `text_combined` column that will be used for TF-IDF. Combining both gives the model a richer signal about each paper's topic.

In [ ]:
import re
import pandas as pd

pd.options.mode.string_storage = "pyarrow"

def clean_series(series):
    return (
        series.fillna("")
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

cleaned_title = clean_series(combined_df['title'])
cleaned_abstract = clean_series(combined_df['abstract'])
cleaned_venue = clean_series(combined_df['venue'])
# Convert list of authors to a single string, then clean it
authors_as_string = combined_df['authors'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
cleaned_authors = clean_series(authors_as_string)

# Performance optimization: Combine text in a
# single vectorized step to avoid multiple Series allocations
combined_df["text_combined"] = (
    cleaned_title
    + " "
    + cleaned_title
    + " "
    + cleaned_abstract
    + " "
    + cleaned_authors
    + " "
    + cleaned_venue
).str.strip()

del cleaned_title, cleaned_abstract, cleaned_authors, authors_as_string

print("\nSample combined text (first 200 chars):")
print(combined_df["text_combined"].iloc[0][:300])

In [ ]:
from collections import defaultdict
import random

graph = defaultdict(list)

print("Building graph.")
for _, row in combined_df.iterrows():
    src = row["id"]

    for dst in row["references"]:
        graph[src].append(dst)
print("Graph complete.")

def random_walk(start, length=10):
    walk = [start]

    for _ in range(length - 1):
        cur = walk[-1]

        if cur in graph and graph[cur]:
            walk.append(random.choice(graph[cur]))
        else:
            break

    return walk

walks = []

print("Generating walks in graph.")
for node in list(graph.keys()):
    for _ in range(2):  # number of walks per node
        walks.append(random_walk(node))
print("Walks successfully generated")

del graph
gc.collect()

In [ ]:
from gensim.models import Word2Vec
from os import cpu_count

print("Running model...")
model = Word2Vec(
    sentences=walks,
    vector_size=20,
    window=5,
    min_count=1,
    negative=5,
    sg=1,
    workers=cpu_count()
)
print("Model finished.")

print("Retrieving embeddings from model.")
emb_df = pd.DataFrame(
    model.wv.vectors,
    index=model.wv.index_to_key
).reset_index()

# Rename the 'index' column to 'id'
emb_df.rename(columns={"index": "id"}, inplace=True)

# Rename the embedding columns (0, 1, 2, ...) to emb_0, emb_1, ...
# Get the list of current column names, excluding 'id'
embedding_cols = [col for col in emb_df.columns if col != 'id']

# Create a dictionary for renaming
rename_dict = {col: f"references_{col}" for col in embedding_cols}
emb_df.rename(columns=rename_dict, inplace=True)

print("Finished retrieving embeddings from model")

display(emb_df)

del model, walks
gc.collect()

In [ ]:
import numpy as np

print("Calculating author count for each paper...")
# Calculate author count directly. This will be a dense Series.
author_counts_series = combined_df['authors'].apply(len)

combined_df['n_authors'] = author_counts_series

display(author_counts_series)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_data = combined_df["text_combined"].reset_index(drop=True)

print(f"Papers used for TF-IDF (have abstract): {len(combined_df):,}")

tfidf_vectorizer = TfidfVectorizer(
    max_features=10_000,
    stop_words="english",
    min_df=3,
    max_df=0.85,
    dtype="float32",
    token_pattern=r'(?u)\b[a-z]{2,}\b'  # letters only
)

X_tfidf = tfidf_vectorizer.fit_transform(text_data)

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"  - Rows: {X_tfidf.shape[0]:,} papers")
print(f"  - Columns: {X_tfidf.shape[1]:,} word features")

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Sample TF-IDF features (words): {list(feature_names[:20])}")

del feature_names, text_data
gc.collect()

In [ ]:
# Merge emb_df into combined_df on the 'id' column
print("Merging reference embeddings into combined_df...")
combined_df = pd.merge(combined_df, emb_df, on="id", how="left")
print("Merge complete. Displaying head of combined_df with new columns:")
display(combined_df.head())

del emb_df
gc.collect()

In [ ]:
print("Filling missing embedding values with 0...")
# Get all columns that start with 'references_'
embedding_cols = [col for col in combined_df.columns if col.startswith('references_')]
combined_df[embedding_cols] = combined_df[embedding_cols].fillna(0)
print("Missing embedding values filled.")

print("Verifying no more missing values in embedding columns:")
print(combined_df[embedding_cols].isnull().sum().sum())

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = (['n_authors', 'n_citation', 'year']
    + [col for col in combined_df.columns if col.startswith('references_')])

df_numerical = combined_df[numerical_cols]

scaler = StandardScaler()

df_numerical_scaled = scaler.fit_transform(df_numerical)

combined_df[numerical_cols] = df_numerical_scaled

display(combined_df[numerical_cols].head())


In [ ]:
import joblib

# Save DataFrame
combined_df.to_parquet(f"{SAVE_DIR}/dblp_preprocessed.parquet", index=False)
# Save TF-IDF stuff
joblib.dump(X_tfidf, f"{SAVE_DIR}/tfidf_matrix.pkl")
joblib.dump(tfidf_vectorizer, f"{SAVE_DIR}/tfidf_vectorizer.pkl")

print("Saved:")
print(f"  dblp_preprocessed.parquet  — {len(combined_df):,} rows, {combined_df.shape[1]} columns")
print(f"  tfidf_matrix.pkl           — {X_tfidf.shape[0]:,} papers x {X_tfidf.shape[1]:,} features")
print(f"  tfidf_vectorizer.pkl       — fitted TfidfVectorizer")